In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import numpy as np
from pathlib import Path
from collections import defaultdict

from eval_fluent_deeponet import (
    make_plot_dict_from_iterative_result,
    select_sample_indices_by_case_id,
    infer_edge_layout,
    PhysicsInterfaceConfig,
    physics_unknown_interface_inference,
    SNIInterfaceConfig,
    sni_unknown_interface_inference,
)
from fluent_deeponet import DeepONet, FeatureNormalizer
from deeponet_fluent_dataset import build_fluent_deeponet_dataset

from plot import plot_prediction_imshow_from_points

In [ ]:
device = torch.device("cuda:3")
ckpt_path = Path("results") / "061626_2" / "checkpoint.pt"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model = DeepONet(**ckpt["model_config"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

y_normalizer = FeatureNormalizer.from_state_dict(ckpt["y_normalizer"]).to(device)
local_aspect_mean = float(ckpt["local_aspect_mean"])
local_aspect_std = float(ckpt["local_aspect_std"])


In [ ]:
FIELD_MAP = {"pressure": "SV_P", "u": "SV_U", "v": "SV_V"}
ROOT_DIR = Path("/home/hantianl/Documents/PIDIF/")
DATASET_NAME = "channel_water_ablation"

def case_paths(ch):
    return {
        "design": ROOT_DIR / f"2d_geometry_specs/{DATASET_NAME}/{ch}.json",
        "mesh": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/{ch}.msh.h5",
        "dat": ROOT_DIR / f"runs_2d/{DATASET_NAME}/{ch}/case2d.dat.h5",
    }

available_cases = [
    ch.name for ch in (ROOT_DIR / "runs_2d" / DATASET_NAME).iterdir()
    if all(case_paths(ch.name)[k].exists() for k in ("design", "mesh", "dat"))
]
case_files = {ch: case_paths(ch) for ch in available_cases}

bc_kwargs = {
    "inlet_v": 0.0,
    "outlet_p": 0.0,
    "wall_u": 0.0,
    "wall_v": 0.0,
}

# TEST_CH = ['channel_18', 'channel_22', 'channel_28', 'channel_31', 'channel_33', 'channel_45', 'channel_51', 'channel_71', 
#            'channel_81', 'channel_89', 'channel_90', 'channel_97', 'channel_111', 'channel_117', 'channel_118', 'channel_133', 
#            'channel_151', 'channel_172', 'channel_193', 'channel_199']
# TEST_CH = ['channel_00', 'channel_01', 'channel_02', 'channel_03', 'channel_04', 'channel_05', 'channel_06', 'channel_07', 
#            'channel_08', 'channel_09']
TEST_CH = [f"channel_{i:03d}" for i in range(50)]  # Test on channels 0-49
rng = np.random.default_rng(0)

test_data = build_fluent_deeponet_dataset(
    case_files=case_files,
    case_ids=TEST_CH,
    n_subdomains="adaptive",
    n_interface_points=256,
    n_boundary_points=256,
    interface_placement="fixed",
    interface_jitter=0,
    insert_sharp_control_point_interfaces=False,
    field_map=FIELD_MAP,
    bc_kwargs=bc_kwargs,
    keep_raw_case_data=False,
    rng=rng,
)

sample_test_ids = select_sample_indices_by_case_id(test_data, case_id=TEST_CH[0])

# Physics inference

In [ ]:
results = defaultdict(dict)
plot_dicts = defaultdict(dict)
for ch in TEST_CH:
    print(f"Physics inference for {ch}")
    sample_test_ids = select_sample_indices_by_case_id(test_data, case_id=ch)

    samples = [test_data["samples"][i] for i in sample_test_ids]
    metadata = [test_data["metadata"][i] for i in sample_test_ids]

    for seed in rng.choice(range(1000), size=5, replace=False):
        print(f"Seed: {seed}")
        phys_config = PhysicsInterfaceConfig(
            max_iter=50,
            lr=1e-2,
            optimizer="lbfgs",
            tol=1e-6,
            random_seed=seed,
            init_mode="fixed",
            init_value=[0.0, 0.0, 0.0],  # [pressure, u, v] for 3-output model
            init_noise_std=0.02,
            optimize_fields=["pressure", "u", "v"],
            viscosity=1.003E-3,
            length_unit_scale=1e-3,  # metadata unit is mm; converts to m
            alpha_traction=0.1,
            alpha_flux=0.3,
            alpha_dirichlet=10,
            alpha_smooth=1e-4,
            alpha_value_l2=0,
            alpha_p=10,
            alpha_u=3,
            alpha_v=0.1,
            optimize_pressure_offsets=False,
            query_batch_size=32768,
            verbose=True,
            verbose_every=25,
        )

        result_phys = physics_unknown_interface_inference(
            model=model,
            samples=samples,
            branch_channel_names=test_data["branch_channel_names"],
            output_channel_names=test_data["output_channel_names"],
            device=device,
            y_normalizer=y_normalizer,
            metadata=metadata,
            local_aspect_mean=local_aspect_mean,
            local_aspect_std=local_aspect_std,
            config=phys_config,
        )
        results[ch][seed] = result_phys

        print("Iterations:", result_phys["n_iter"])
        print("Final physics losses:", result_phys["physics_loss_history"][-1])

        plot_dict = make_plot_dict_from_iterative_result(
            result_phys,
            test_data,
            sample_test_ids,
            output_channel_names=test_data["output_channel_names"],
        )
        plot_dicts[ch][seed] = plot_dict

        pred = plot_dict["pred"]
        truth = plot_dict["truth"]
        err = np.abs(pred - truth)

        rel_l2 = np.linalg.norm(err, axis=0) / np.linalg.norm(truth, axis=0)
        print(f"Relative L2 error: {rel_l2}, mean: {np.mean(rel_l2)}")
        print(f"Max absolute error: {np.abs(plot_dict['pred'] - plot_dict['truth']).max(axis=0)}")

In [ ]:
pd['pred']

In [ ]:
errs = []
for ch, seeds in plot_dicts.items():
    err_geo = []
    p_area = []
    print(ch)
    for seed, pd in seeds.items():
        pred = pd["pred"]
        truth = pd["truth"]
        area = (pd["x"] > 0.1) & (pd["x"] < 0.12) & (pd["y"] > 0.04) & (pd["y"] < 0.06)
        p = pd["pred"][area, 0].mean()
        p_area.append(p)
        err = np.abs(pred - truth)
        print(p)
        rel_l2 = np.linalg.norm(err, axis=0) / np.linalg.norm(truth, axis=0)
        err_geo.append(rel_l2)

    p_area = np.array(p_area)
    err_geo = np.array(err_geo)
    p_mean = p_area.mean()
    p_std = p_area.std()
    inliers = np.abs(p_area - p_mean) < p_std
    err_geo = err_geo[inliers]
    errs.append(err_geo.mean(axis=0))

errs = np.array(errs)
errs.mean(axis=0)

In [ ]:
output_channel_names = test_data["output_channel_names"]
for ch in TEST_CH[2:3]:
    # output_dir = Path("results") / "physics_inference_cp_metis" / ch
    # output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Plotting {ch}")
    metadatas = [test_data["metadata"][i] for i in range(len(test_data["metadata"])) if test_data["metadata"][i]["case_id"] == ch]
    for field_name in output_channel_names:
        field_idx = output_channel_names.index(field_name)
        plot_prediction_imshow_from_points(
            x=plot_dicts[ch]["x"],
            y=plot_dicts[ch]["y"],
            pred=plot_dicts[ch]["pred"],
            truth=plot_dicts[ch]["truth"],
            field_name=field_name,
            field_idx=field_idx,
            n_y_plot=200,
            metadata=metadatas, 
            mesh_h5=case_files[ch]["mesh"],
            # output_dir=output_dir,
        )

In [ ]:
output_channel_names = test_data["output_channel_names"]
int_id = 1
for val_id in range(3):
    br = test_data["samples"][sample_test_ids[int_id]]["branch"]
    real_t = br[(br[:, 0] == 0) & (br[:, 3] == 1)][:, val_id + 4]
    br_pred = result_phys["branch_final"][int_id]
    pred_t = br_pred[(br_pred[:, 0] == 0) & (br_pred[:, 3] == 1)][:, val_id + 4]
    pred_left = y_normalizer.decode(result_phys["pred_left_interface"]).cpu().numpy()
    pred_right = y_normalizer.decode(result_phys["pred_right_interface"]).cpu().numpy()
    pred_left = pred_left[int_id, :, val_id]
    pred_right = pred_right[int_id, :, val_id]


    plt.figure(figsize=(8, 6))
    plt.plot(real_t, zorder=10, lw=2, label="Real value", alpha=0.7)
    plt.plot(pred_t, label="Inferred interface")
    plt.plot(pred_left, label="Predicted boundary value left")
    plt.plot(pred_right, label="Predicted boundary value right")
        
    plt.xlabel("point index (0 is bottom, 256 is top)", fontsize=16)
    plt.ylabel(f"{output_channel_names[val_id]}", fontsize=16)
    plt.title(f"Physics inference (interface {int_id})")
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    leg = plt.legend(loc=1, fontsize=13)
    leg.set_zorder(20)
    plt.show()


# Physics inference weight sweep

Sweep traction / flux / Dirichlet loss weights, save metrics, field predictions, interface conditions, and plots for each combination.

In [ ]:
import itertools
import json
from dataclasses import replace

sweep_samples = [test_data["samples"][i] for i in sample_test_ids]
sweep_metadata = [test_data["metadata"][i] for i in sample_test_ids]
sweep_output_channel_names = list(test_data["output_channel_names"])
sweep_channel_labels = ["pressure", "u-velocity", "v-velocity"]

SWEEP_BASE_PHYS_CONFIG = PhysicsInterfaceConfig(
    max_iter=50,
    lr=1e-2,
    optimizer="lbfgs",
    tol=1e-6,
    random_seed=0,
    init_mode="fixed",
    init_value=[0.0, 0.0, 0.0],
    init_noise_std=0.0,
    optimize_fields=["pressure", "u", "v"],
    viscosity=1.003E-3,
    length_unit_scale=1e-3,
    alpha_traction=0.1,
    alpha_flux=0.3,
    alpha_dirichlet=10.0,
    alpha_smooth=1e-4,
    alpha_value_l2=0,
    optimize_pressure_offsets=False,
    query_batch_size=32768,
    verbose=False,
    verbose_every=25,
)

SWEEP_ALPHA_VALUES = [0.1, 0.3, 1, 3, 10.0]
SWEEP_WEIGHT_COMBINATIONS = list(
    itertools.product(SWEEP_ALPHA_VALUES, SWEEP_ALPHA_VALUES, SWEEP_ALPHA_VALUES)
)

SWEEP_OUTPUT_ROOT = Path("results") / "physics_bc_weight_cp" / TEST_CH[0]
SWEEP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Ground-truth interior interface values (shared across all weight runs).
sweep_true_branch = np.stack(
    [test_data["samples"][i]["branch"] for i in sample_test_ids], axis=0
).astype(np.float32)
sweep_layout = infer_edge_layout(sweep_true_branch[0], test_data["branch_channel_names"])
sweep_value_idx = sweep_layout.value_channels
sweep_n_sub = sweep_true_branch.shape[0]

sweep_true_interfaces = []
for interface_id in range(1, sweep_n_sub):
    sweep_true_interfaces.append(
        sweep_true_branch[interface_id][np.ix_(sweep_layout.left, sweep_value_idx)]
    )
sweep_true_interfaces = np.stack(sweep_true_interfaces, axis=0)

print(f"Running {len(SWEEP_WEIGHT_COMBINATIONS)} weight combinations -> {SWEEP_OUTPUT_ROOT}")

In [ ]:
def _sweep_infer_interfaces_from_branch(infer_branch: np.ndarray) -> np.ndarray:
    interfaces = []
    for interface_id in range(1, sweep_n_sub):
        i_right = infer_branch[interface_id - 1][np.ix_(sweep_layout.right, sweep_value_idx)]
        i_left = infer_branch[interface_id][np.ix_(sweep_layout.left, sweep_value_idx)]
        interfaces.append(0.5 * (i_right + i_left))
    return np.stack(interfaces, axis=0)


def _sweep_weight_run_name(a: float, b: float, c: float) -> str:
    return (
        f"p_{a:g}_u_{b:g}_v_{c:g}"
    )


def _sweep_save_interface_profile_plots(result_phys, run_dir: Path, int_id: int = 1) -> None:
    iface_plot_dir = run_dir / "plots" / "interfaces"
    iface_plot_dir.mkdir(parents=True, exist_ok=True)

    br = test_data["samples"][sample_test_ids[int_id]]["branch"]
    br_pred = result_phys["branch_final"][int_id]
    pred_left = y_normalizer.decode(result_phys["pred_left_interface"]).cpu().numpy()
    pred_right = y_normalizer.decode(result_phys["pred_right_interface"]).cpu().numpy()

    for val_id, field_name in enumerate(sweep_output_channel_names):
        real_t = br[(br[:, 0] == 0) & (br[:, 3] == 1)][:, val_id + 4]
        pred_t = br_pred[(br_pred[:, 0] == 0) & (br_pred[:, 3] == 1)][:, val_id + 4]
        pred_left_line = pred_left[int_id, :, val_id]
        pred_right_line = pred_right[int_id, :, val_id]

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot(real_t, zorder=10, lw=2, label="Real value", alpha=0.7)
        ax.plot(pred_t, label="Inferred interface")
        ax.plot(pred_left_line, label="Predicted boundary value left")
        ax.plot(pred_right_line, label="Predicted boundary value right")
        ax.set_xlabel("point index (0 is bottom, 256 is top)", fontsize=16)
        ax.set_ylabel(field_name, fontsize=16)
        ax.set_title(f"Physics inference (interface {int_id})")
        ax.legend(loc=1, fontsize=13)
        fig.tight_layout()
        fig.savefig(
            iface_plot_dir / f"interface{int_id}_{field_name}.png",
            dpi=200,
            bbox_inches="tight",
        )
        plt.close(fig)


def _sweep_save_interface_error_plots(
    infer_interfaces: np.ndarray,
    run_dir: Path,
    run_name: str,
    int_eval: int = 0,
) -> None:
    yn_mean = y_normalizer.mean.detach().cpu().numpy().astype(np.float32).reshape(1, 1, -1)
    yn_std = y_normalizer.std.detach().cpu().numpy().astype(np.float32).reshape(1, 1, -1)
    yn_std = np.maximum(yn_std, 1.0e-12)

    true_norm = (sweep_true_interfaces - yn_mean) / yn_std
    infer_norm = (infer_interfaces - yn_mean) / yn_std
    abs_err_norm = np.abs(infer_norm - true_norm)

    iface_plot_dir = run_dir / "plots" / "interfaces"
    iface_plot_dir.mkdir(parents=True, exist_ok=True)

    mae_if_ch = abs_err_norm.mean(axis=1)
    fig, ax = plt.subplots(figsize=(9, 4))
    for ch in range(mae_if_ch.shape[1]):
        ax.plot(
            np.arange(1, mae_if_ch.shape[0] + 1),
            mae_if_ch[:, ch],
            marker="o",
            label=sweep_channel_labels[ch],
        )
    ax.set_xlabel("Interior interface index")
    ax.set_ylabel("MAE (normalized)")
    ax.set_title(f"{run_name}: interface error by interface")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(iface_plot_dir / "interface_mae_by_index.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(3, 1, figsize=(8, 2.2 * 4), sharex=True)
    for ch, ax in enumerate(axes):
        ax.plot(sweep_true_interfaces[int_eval, :, ch], lw=2, label="Truth")
        ax.plot(infer_interfaces[int_eval, :, ch], lw=2, label="Inferred")
        ax.set_ylabel(sweep_channel_labels[ch])
        ax.legend(loc=1)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel("point index (0 is bottom, 256 is top)")
    fig.suptitle(f"{run_name}: interface {int_eval} profiles", y=1.02)
    fig.tight_layout()
    fig.savefig(
        iface_plot_dir / f"interface{int_eval}_profiles.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close(fig)


sweep_summary_rows = []

for run_idx, (alpha_p, alpha_u, alpha_v) in enumerate(SWEEP_WEIGHT_COMBINATIONS):
    run_name = _sweep_weight_run_name(alpha_p, alpha_u, alpha_v)
    run_dir = SWEEP_OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    sweep_phys_config = replace(
        SWEEP_BASE_PHYS_CONFIG,
        alpha_p=alpha_p,
        alpha_u=alpha_u,
        alpha_v=alpha_v,
        verbose=(run_idx == 0),
    )

    print(
        f"[{run_idx + 1}/{len(SWEEP_WEIGHT_COMBINATIONS)}] "
        f"p={alpha_p:g}, u={alpha_u:g}, v={alpha_v:g}"
    )

    sweep_result = physics_unknown_interface_inference(
        model=model,
        samples=sweep_samples,
        branch_channel_names=test_data["branch_channel_names"],
        output_channel_names=sweep_output_channel_names,
        device=device,
        y_normalizer=y_normalizer,
        metadata=sweep_metadata,
        local_aspect_mean=local_aspect_mean,
        local_aspect_std=local_aspect_std,
        config=sweep_phys_config,
    )

    sweep_plot_dict = make_plot_dict_from_iterative_result(
        sweep_result,
        test_data,
        sample_test_ids,
        output_channel_names=sweep_output_channel_names,
    )

    pred = sweep_plot_dict["pred"]
    truth = sweep_plot_dict["truth"]
    err = np.abs(pred - truth)
    rel_l2 = np.linalg.norm(err, axis=0) / np.maximum(np.linalg.norm(truth, axis=0), 1.0e-12)
    overall_avg_rel_l2 = float(np.mean(rel_l2))
    max_field_err = err.max(axis=0)

    infer_branch = np.asarray(sweep_result["branch_final"], dtype=np.float32)
    infer_interfaces = _sweep_infer_interfaces_from_branch(infer_branch)
    pred_left = y_normalizer.decode(sweep_result["pred_left_interface"]).cpu().numpy()
    pred_right = y_normalizer.decode(sweep_result["pred_right_interface"]).cpu().numpy()

    final_losses = sweep_result["physics_loss_history"][-1]
    if hasattr(final_losses, "items"):
        final_losses = {k: float(v) for k, v in final_losses.items()}

    metrics = {
        "run_name": run_name,
        "alpha_p": float(alpha_p),
        "alpha_u": float(alpha_u),
        "alpha_v": float(alpha_v),
        "converged": bool(sweep_result["converged"]),
        "n_iter": int(sweep_result["n_iter"]),
        "rel_l2": {
            name: float(rel_l2[i]) for i, name in enumerate(sweep_output_channel_names)
        },
        "overall_avg_rel_l2": overall_avg_rel_l2,
        "max_field_error": {
            name: float(max_field_err[i]) for i, name in enumerate(sweep_output_channel_names)
        },
        "final_physics_losses": final_losses,
    }

    with open(run_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    np.savez_compressed(
        run_dir / "field_predictions.npz",
        pred=pred.astype(np.float32),
        truth=truth.astype(np.float32),
        x=sweep_plot_dict["x"].astype(np.float32),
        y=sweep_plot_dict["y"].astype(np.float32),
        rel_l2=rel_l2.astype(np.float32),
        overall_avg_rel_l2=np.asarray(overall_avg_rel_l2, dtype=np.float32),
        max_field_error=max_field_err.astype(np.float32),
        output_channel_names=np.asarray(sweep_output_channel_names),
    )

    np.savez_compressed(
        run_dir / "interface_conditions.npz",
        true_interfaces=sweep_true_interfaces.astype(np.float32),
        infer_interfaces=infer_interfaces.astype(np.float32),
        branch_final=infer_branch,
        pred_left_interface=pred_left.astype(np.float32),
        pred_right_interface=pred_right.astype(np.float32),
    )

    plot_dir = run_dir / "plots" / "fields"
    for field_name in sweep_output_channel_names:
        field_idx = sweep_output_channel_names.index(field_name)
        plot_prediction_imshow_from_points(
            x=sweep_plot_dict["x"],
            y=sweep_plot_dict["y"],
            pred=sweep_plot_dict["pred"],
            truth=sweep_plot_dict["truth"],
            field_name=field_name,
            field_idx=field_idx,
            n_y_plot=200,
            metadata=test_data["metadata"],
            mesh_h5=case_files[TEST_CH[0]]["mesh"],
            output_dir=str(plot_dir),
            filename_prefix="field",
            show=False,
        )

    _sweep_save_interface_profile_plots(sweep_result, run_dir, int_id=1)
    _sweep_save_interface_error_plots(infer_interfaces, run_dir, run_name, int_eval=0)

    sweep_summary_rows.append(metrics)
    
    print(f"  overall_avg_rel_l2: {overall_avg_rel_l2:.4e}")
    print(
        "  rel_l2:",
        {name: f"{metrics['rel_l2'][name]:.4e}" for name in sweep_output_channel_names},
    )
    print(
        "  max field err:",
        {name: f"{metrics['max_field_error'][name]:.4e}" for name in sweep_output_channel_names},
    )

with open(SWEEP_OUTPUT_ROOT / "summary.json", "w", encoding="utf-8") as f:
    json.dump(sweep_summary_rows, f, indent=2)

print(f"Saved sweep results to {SWEEP_OUTPUT_ROOT}")

In [ ]:
# Quick summary table across all weight combinations.
print(f"{'run_name':<45} {'avg rel_l2':<12} {'rel_l2 (p,u,v)':<40} {'max err (p,u,v)':<40}")
print("-" * 137)
for row in sweep_summary_rows:
# for row in summary:
    rel = row["rel_l2"]
    mx = row["max_field_error"]
    rel_str = ", ".join(f"{rel[n]:.3e}" for n in sweep_output_channel_names)
    mx_str = ", ".join(f"{mx[n]:.3e}" for n in sweep_output_channel_names)
    avg_rel_l2 = row.get("overall_avg_rel_l2", float(np.mean(list(rel.values()))))
    print(f"{row['run_name']:<45} {avg_rel_l2:<12.3e} {rel_str:<40} {mx_str:<40}")

In [ ]:
min_overall = (float("inf"), float("inf"), float("inf"), None)
min_pressure = (float("inf"), float("inf"), float("inf"), None)
min_u = (float("inf"), float("inf"), float("inf"), None)
min_v = (float("inf"), float("inf"), float("inf"), None)
summary = json.load(open(SWEEP_OUTPUT_ROOT / "summary.json", "r"))
for run in summary:
    l2 = run["rel_l2"]
    if l2["pressure"] < min_pressure[0]:
        min_pressure = (l2["pressure"], l2["u"], l2["v"], run["run_name"])
    if l2["u"] < min_u[1]:
        min_u = (l2["pressure"], l2["u"], l2["v"], run["run_name"])
    if l2["v"] < min_v[2]:
        min_v = (l2["pressure"], l2["u"], l2["v"], run["run_name"])
    if sum(l2.values()) < sum(min_overall[:3]):
        min_overall = (l2["pressure"], l2["u"], l2["v"], run["run_name"])
print("min_overall", min_overall)
print("min_pressure", min_pressure)
print("min_u", min_u)
print("min_v", min_v)

In [ ]:
# Optional: inspect one saved sweep run (change run_name to compare combinations).
sweep_inspect_run_name = _sweep_weight_run_name(10, 3, 0.1)
sweep_inspect_dir = SWEEP_OUTPUT_ROOT / sweep_inspect_run_name

with open(sweep_inspect_dir / "metrics.json", encoding="utf-8") as f:
    sweep_inspect_metrics = json.load(f)

sweep_field_data = np.load(sweep_inspect_dir / "field_predictions.npz", allow_pickle=True)
sweep_iface_data = np.load(sweep_inspect_dir / "interface_conditions.npz")

print("Inspecting:", sweep_inspect_run_name)
print("overall_avg_rel_l2:", sweep_inspect_metrics.get("overall_avg_rel_l2"))
print("rel_l2:", sweep_inspect_metrics["rel_l2"])
print("max_field_error:", sweep_inspect_metrics["max_field_error"])
print("final_physics_losses:", sweep_inspect_metrics["final_physics_losses"])

for field_name in sweep_output_channel_names:
    field_idx = sweep_output_channel_names.index(field_name)
    plot_prediction_imshow_from_points(
        x=sweep_field_data["x"],
        y=sweep_field_data["y"],
        pred=sweep_field_data["pred"],
        truth=sweep_field_data["truth"],
        field_name=field_name,
        field_idx=field_idx,
        n_y_plot=200,
        metadata=test_data["metadata"],
        mesh_h5=case_files[TEST_CH[0]]["mesh"],
    )

In [ ]:
# Optional: interface profile plot for the same inspected sweep run.
sweep_int_eval = 5
sweep_true_if = sweep_iface_data["true_interfaces"]
sweep_infer_if = sweep_iface_data["infer_interfaces"]

fig, axes = plt.subplots(3, 1, figsize=(8, 2.2 * 4), sharex=True)
for ch, ax in enumerate(axes):
    ax.plot(sweep_true_if[sweep_int_eval, :, ch], lw=2, label="Truth")
    ax.plot(sweep_infer_if[sweep_int_eval, :, ch], lw=2, label="Inferred")
    ax.set_ylabel(sweep_channel_labels[ch])
    ax.legend(loc=1)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("point index (0 is bottom, 256 is top)")
fig.suptitle(f"{sweep_inspect_run_name}: interface {sweep_int_eval}", y=1.02)
fig.tight_layout()
plt.show()

# SNI (Schwarz Neural Iteration) inference

Same setup as the physics inference above, but the interior interface values are
solved with a relaxed **Schwarz fixed-point iteration** (Schwarz Neural
Inference, arXiv:2504.00510) instead of the physics-based optimizer. Each
iteration runs the frozen DeepONet on every subdomain, evaluates the two
predicted traces on each shared interface, and updates the interface as their
relaxed average `z <- (1 - tau) * z + tau * 0.5 * (u_left + u_right)` until the
interface mismatch stagnates.

In [ ]:
from eval_fluent_deeponet import (
    SNIInterfaceConfig,
    sni_unknown_interface_inference,
)

sni_results = {}
sni_plot_dicts = {}
for ch in TEST_CH:
    print(f"SNI inference for {ch}")
    sample_test_ids = select_sample_indices_by_case_id(test_data, case_id=ch)

    samples = [test_data["samples"][i] for i in sample_test_ids]
    metadata = [test_data["metadata"][i] for i in sample_test_ids]
    sni_config = SNIInterfaceConfig(
        max_iter=500,
        tau=0.5,                       # Schwarz relaxation in (0, 1]
        tol=1e-6,
        random_seed=0,
        init_mode="fixed",
        init_value=[0.0, 0.0, 0.0],    # [pressure, u, v] for 3-output model
        init_noise_std=0.0,
        optimize_fields=["pressure", "u", "v"],
        stagnation_window=10,
        stagnation_decimals=6,
        query_batch_size=32768,
        verbose=True,
        verbose_every=25,
    )

    result_sni = sni_unknown_interface_inference(
        model=model,
        samples=samples,
        branch_channel_names=test_data["branch_channel_names"],
        output_channel_names=test_data["output_channel_names"],
        device=device,
        y_normalizer=y_normalizer,
        metadata=metadata,
        local_aspect_mean=local_aspect_mean,
        local_aspect_std=local_aspect_std,
        config=sni_config,
    )
    sni_results[ch] = result_sni

    print("Iterations:", result_sni["n_iter"], "| converged:", result_sni["converged"])
    print("Final interface mse:", result_sni["metric_history"][-1] if result_sni["metric_history"] else float("nan"))

    plot_dict = make_plot_dict_from_iterative_result(
        result_sni,
        test_data,
        sample_test_ids,
        output_channel_names=test_data["output_channel_names"],
    )
    sni_plot_dicts[ch] = plot_dict

    pred = plot_dict["pred"]
    truth = plot_dict["truth"]
    err = np.abs(pred - truth)

    rel_l2 = np.linalg.norm(err, axis=0) / np.linalg.norm(truth, axis=0)
    print(f"Relative L2 error: {rel_l2}, mean: {np.mean(rel_l2)}")
    print(f"Max absolute error: {np.abs(plot_dict['pred'] - plot_dict['truth']).max(axis=0)}")

In [ ]:
# Aggregate SNI rel-L2 across seeds/cases (mirrors the physics aggregation cell).
sni_errs = []
for ch, pd in sni_plot_dicts.items():
    pred = pd["pred"]
    truth = pd["truth"]
    err = np.abs(pred - truth)
    rel_l2 = np.linalg.norm(err, axis=0) / np.linalg.norm(truth, axis=0)

    sni_errs.append(rel_l2)

sni_errs = np.array(sni_errs)
sni_errs.mean(axis=0)

In [ ]:
# Interface profile check for the SNI solution (mirrors the physics interface plot).
output_channel_names = test_data["output_channel_names"]
int_id = 1
for val_id in range(3):
    br = test_data["samples"][sample_test_ids[int_id]]["branch"]
    real_t = br[(br[:, 0] == 0) & (br[:, 3] == 1)][:, val_id + 4]
    br_pred = result_sni["branch_final"][int_id]
    pred_t = br_pred[(br_pred[:, 0] == 0) & (br_pred[:, 3] == 1)][:, val_id + 4]
    pred_left = y_normalizer.decode(result_sni["pred_left_interface"]).cpu().numpy()
    pred_right = y_normalizer.decode(result_sni["pred_right_interface"]).cpu().numpy()
    pred_left = pred_left[int_id, :, val_id]
    pred_right = pred_right[int_id, :, val_id]

    plt.figure(figsize=(8, 6))
    plt.plot(real_t, zorder=10, lw=2, label="Real value", alpha=0.7)
    plt.plot(pred_t, label="Inferred interface (SNI)")
    plt.plot(pred_left, label="Predicted boundary value left")
    plt.plot(pred_right, label="Predicted boundary value right")

    plt.xlabel("point index (0 is bottom, 256 is top)", fontsize=16)
    plt.ylabel(f"{output_channel_names[val_id]}", fontsize=16)
    plt.title(f"SNI inference (interface {int_id})")
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    leg = plt.legend(loc=1, fontsize=13)
    leg.set_zorder(20)
    plt.show()